#### Stream Autoloader

##### readStream *schema can be enforced on read
- based on the dataset already in the "folder" or query specific datatypes;
- inferColumnTypes
- timestamp
- checkpointLocation
- trigger(AvailableNow=True)
  - multiple triggers

In [0]:
customers_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/medallion_catalog/db_landing/vol_medallion/customer_autoloader/")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
    .load("/Volumes/medallion_catalog/db_landing/vol_medallion/customer_autoloader/")
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

##### New columns:: file_path, ingestion_date

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

##### writeStream

- Idempotent Sinks enables exactly once guarantees

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/medallion_catalog/db_landing/vol_medallion/customer_autoloader/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("medallion_catalog.db_bronze.customer_autoloader")
)

Autoloader differently than spark streaming will insure no duplicates; it will only load new datasets or updates old version as needed;

In [0]:
%sql

SELECT * FROM medallion_catalog.db_bronze.customer_autoloader

- Stop the streaming process.
- Terminate the cluster.

In [0]:
# streaming_query.stop()
